# People’s Daily Page 1–3 Priority Classifier

This notebook contains the final training and analysis workflow for a time-aware Multi-BERT classifier. The model predicts whether a People’s Daily article belongs to Page 1–3 priority placement.

**GitHub/local note:** this notebook no longer assumes Google Drive by default. Set `BASE_DIR` to the directory containing the data files before running.


## 1. Environment Setup and Path Configuration

Use this cell to configure the project root directory. In Colab, Google Drive mounting is optional. In GitHub or local environments, set `BASE_DIR` directly.


In [ ]:
import os

# Optional Google Drive mount for Colab users.
# In GitHub/local runs, skip this cell or leave USE_GOOGLE_DRIVE unset.
USE_GOOGLE_DRIVE = os.environ.get("USE_GOOGLE_DRIVE", "0") == "1"

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True)
        DEFAULT_BASE_DIR = "/content/drive/MyDrive/Colab Notebooks"
    except Exception as e:
        print(f"Google Drive mount failed or is unavailable: {e}")
        DEFAULT_BASE_DIR = "."
else:
    DEFAULT_BASE_DIR = "."

# For GitHub/local use, set BASE_DIR to the directory containing:
# train_df.parquet, val_df 1.parquet, test_df 1.parquet, and outputs/.
BASE_DIR = os.environ.get("BASE_DIR", DEFAULT_BASE_DIR)
os.environ["BASE_DIR"] = BASE_DIR

print(f"BASE_DIR = {BASE_DIR}")

expected_train_path = os.path.join(BASE_DIR, "train_df.parquet")
if os.path.exists(expected_train_path):
    print(f"Found training file: {expected_train_path}")
else:
    print(f"Training file not found at: {expected_train_path}")
    print("If running locally/GitHub, set BASE_DIR to your data directory.")


## 2. Optional: Download Pretrained Chinese BERT Models

This cell downloads the pretrained model snapshots used by the classifier. Skip this cell if the models are already cached or if you prefer Hugging Face to download them automatically during training.


In [ ]:
from huggingface_hub import snapshot_download
import os

# Models used in this project.
model_ids = [
    "hfl/chinese-roberta-wwm-ext",
    "google-bert/bert-base-chinese",
    "hfl/chinese-bert-wwm-ext",
]

# Store Hugging Face model snapshots under BASE_DIR/hf_models.
base_dir = os.path.join(os.environ.get("BASE_DIR", "."), "hf_models")
os.makedirs(base_dir, exist_ok=True)

for model_id in model_ids:
    local_dir = os.path.join(base_dir, model_id.split("/")[-1])
    print(f"Downloading {model_id} to {local_dir}...")
    snapshot_download(
        repo_id=model_id,
        local_dir=local_dir,
        local_dir_use_symlinks=False,
    )

print("All model snapshots have been downloaded.")


## 3. Main Training Pipeline: Time-Aware Page 1–3 Priority Classifier

This is the main training cell. It constructs Page 1–3 labels, builds political and temporal features, trains three Chinese BERT-family classifiers, and combines them through weighted soft ensemble.


In [ ]:
# =========================================================
# Time-aware Page 1-3 Priority Classifier
# + Multi-BERT Soft Ensemble
# =========================================================

import os
import re
import glob
import json
import random
import signal
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
)

from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup


# =========================================================
# 0. Config
# =========================================================

BASE_DIR = os.environ.get("BASE_DIR", ".")

RUN_ALL_MODELS = True
SINGLE_MODEL_KEY = "roberta_wwm_ext"

OUTPUT_SUFFIX = "page1to3_timeaware_multibert_ensemble"
RESUME_FROM_CHECKPOINT = False

MODEL_CONFIGS = {
    "roberta_wwm_ext": {
        "model_id": "hfl/chinese-roberta-wwm-ext",
        "model_tag": "chinese-roberta-wwm-ext",
        "display_name": "RoBERTa-wwm-ext",
    },
    "bert_base_chinese": {
        "model_id": "google-bert/bert-base-chinese",
        "model_tag": "bert-base-chinese",
        "display_name": "BERT base Chinese",
    },
    "chinese_bert_wwm_ext": {
        "model_id": "hfl/chinese-bert-wwm-ext",
        "model_tag": "chinese-bert-wwm-ext",
        "display_name": "Chinese BERT wwm-ext",
    },
}

RANDOM_SEED = 42

PAGE_COL = "scraped page number"
TEXT_COL = "title"
CONTENT_COL = "body"
DATE_COL = "published_at"
LABEL_COL = "labels"

TRAIN_SPLIT_FILENAME = "train_df.parquet"
VAL_SPLIT_FILENAME = "val_df 1.parquet"
TEST_SPLIT_FILENAME = "test_df 1.parquet"

MAX_LENGTH = 128
NUM_EPOCHS = 8
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 1
WARMUP_RATIO = 0.1
EARLY_STOP_PATIENCE = 2
USE_GRADIENT_CLIPPING = True
MAX_GRAD_NORM = 1.0

USE_FIRST_LAST_SENTENCE = True
USE_TITLE_KEYWORDS = True
USE_BODY_KEYWORDS = True
USE_KEYWORD_INTERACTIONS = True

BODY_KEYWORD_MAX_CHARS = 1200
PREV_FRONT_LOOKBACK_DAYS = 30

SAVE_CHECKPOINT_EVERY_N_EPOCHS = 1
MAX_CHECKPOINTS_TO_KEEP = 3

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Label definition
POSITIVE_PAGE_MAX = 3

# Loss config
USE_FOCAL_LOSS = True
FOCAL_GAMMA = 1.5
POS_CLASS_WEIGHT = 8.0

# Time-decay sample weighting
USE_TIME_DECAY_WEIGHT = True
TIME_DECAY_HALFLIFE_DAYS = 365.0
TIME_DECAY_MIN_WEIGHT = 0.30
TIME_DECAY_MAX_WEIGHT = 1.00

# Ensemble config
ENSEMBLE_WEIGHT_SEARCH_STEP = 0.05


def sanitize_suffix(value: str) -> str:
    cleaned = re.sub(r"[^0-9A-Za-z._-]+", "-", str(value).strip())
    return cleaned.strip("-")


OUTPUT_SUFFIX = sanitize_suffix(OUTPUT_SUFFIX)

GLOBAL_COMPARISON_DIR = os.path.join(
    BASE_DIR,
    "outputs",
    "page1to3_timeaware_multibert_ensemble",
)
os.makedirs(GLOBAL_COMPARISON_DIR, exist_ok=True)


# =========================================================
# 1. Keywords
# =========================================================

TITLE_KEYWORD_CATEGORIES = {
    "top_leadership": [
        "习近平", "总书记", "国家主席", "中央军委主席",
        "李强", "赵乐际", "王沪宁", "蔡奇", "丁薛祥", "薛祥", "李希", "韩正",
        "李克强", "梁强", "阮富仲", "洪森", "苏林", "普京", "黄循财", "卢拉",
        "莫迪", "穆罕默德", "舒斯京", "拉马", "鲁托", "博沃", "欧尔", "莱恩", "桑切斯", "马克"
    ],
    "central_institutions": [
        "中共中央", "党中央", "中央政治局", "中央政治局常委", "中央政治局常委会", "中央军委",
        "国务院", "全国人大", "全国政协", "中共中央办公厅", "中央书记处",
        "中办国办", "中共中央国务院", "全军", "武警部队", "统一战线", "国务院令",
        "常委会", "常务委员会"
    ],
    "leadership_activities": [
        "主持召开", "主持", "出席", "会见", "接见", "视察", "考察", "调研",
        "重要讲话", "重要指示", "重要批示", "情况汇报", "工作汇报", "听取", "座谈",
        "看望", "致辞", "演讲", "主旨", "离京", "出访", "抵达", "回到",
        "春节前夕", "新年贺词", "团拜会", "预备会议", "常务会议", "全体会议",
        "闭幕式", "首发式", "开班式", "欢迎宴会", "晚宴", "出席会议", "隆重举行", "召开"
    ],
    "official_messages": [
        "贺信", "贺电", "致贺", "唁电", "慰问电", "复信", "回信", "致信", "致电",
        "祝贺", "命令", "指示", "批示", "致以", "问候", "勉励", "祝福",
        "署名文章", "发表", "出版发行", "单行本", "读本", "英文版", "社论", "述评", "纪实"
    ],
    "political_meetings": [
        "中央政治局会议", "中央经济工作会议", "中央农村工作会议", "国务院常务会议",
        "全国两会", "全国人民代表大会", "中国人民政治协商会议", "峰会", "全会",
        "代表大会", "开幕会", "联组会", "小组会议", "会议议程", "十四五", "十五五"
    ],
    "diplomacy": [
        "国事访问", "非正式", "外事活动", "欢迎仪式", "欢迎宴会", "欢送", "送行"
    ],
    "policy_or_philosophy": [
        "强国建设", "民族复兴", "坚定信心", "奋发有为", "锲而不舍", "中国式现代化",
        "高质量发展", "改革开放", "共同富裕", "新质生产力", "优秀党员", "党性", "党纪", "论述"
    ],
    "military": [
        "军衔", "上将", "部队", "军队", "战士", "我军", "全军", "动员", "检阅", "分列式"
    ],
    "foreign_entities": [
        "越南共产党", "越共中央", "老挝人民革命党", "亚太经合组织", "欧盟委员会",
        "十国集团", "阿塞拜疆", "斯洛伐克", "斐济", "国王", "总统", "元首", "参议长",
        "财政部长", "国民议会", "参议院"
    ],
    "other_high_freq": [
        "火化", "遗体", "牵挂", "求是", "杂志", "专题学习", "述职", "连任", "就任",
        "学习", "研讨班", "会议精神", "贯彻落实"
    ],
}

BODY_KEYWORD_CATEGORIES = {
    "top_leadership": [
        "习近平", "蔡奇", "韩正", "薛祥", "王沪宁", "李希", "李强", "赵乐际",
        "彭丽媛", "普京", "莫迪", "阮富仲", "苏林", "梁强", "胡锦涛", "李克强"
    ],
    "official_messages": [
        "贺信", "贺电", "致贺", "慰问电", "唁电", "回信", "致电", "祝贺",
        "命令", "致以", "就任", "勉励", "悼念", "沉痛", "火化", "遗像", "生前友好"
    ],
    "central_institutions": [
        "中共中央", "中央办公厅", "中央书记处", "中央政治局常委",
        "中央政治局常委会", "中央军委", "军委", "各级党委", "全国政协",
        "政协全国委员会", "政协常委会", "越南共产党", "越共中央", "武警部队"
    ],
    "state_places": [
        "北京人民大会堂", "人民大会堂", "中南海", "北京中南海",
        "北京钓鱼台国宾馆", "天安门广场", "主席台", "检阅台"
    ],
    "official_activities_discourse": [
        "陪同", "重要讲话", "特命", "诚挚", "团拜会", "党外人士", "主持", "会谈",
        "工作汇报", "情况汇报", "汇报会", "列席会议", "主持会议", "出席会议",
        "专题学习", "读书班", "欢迎仪式", "欢迎宴会", "欢送", "送行",
        "乘专机", "专机", "返京", "出京", "下榻", "红毯", "礼兵", "仪仗队",
        "军乐团", "鸣放", "礼炮", "国歌", "握手", "挥手致意", "接见", "十四五", "十五五"
    ],
    "Announcement_of_the_Passing_of_an_Important_Figure": [
        "火化", "悼念", "沉痛", "遗像", "生前友好"
    ],
}

TITLE_KEYWORD_COMBOS = [
    ("习近平", "重要讲话"),
    ("习近平", "重要指示"),
    ("习近平", "重要批示"),
    ("习近平", "贺信"),
    ("习近平", "回信"),
    ("习近平", "会见"),
    ("习近平", "主持召开"),
    ("中央政治局", "会议"),
    ("中共中央", "国务院"),
    ("中央军委", "习近平"),
    ("李强", "国务院"),
    ("赵乐际", "全国人大"),
    ("王沪宁", "全国政协"),
    ("强国建设", "民族复兴"),
    ("中国式现代化", "高质量发展"),
    ("欢迎仪式", "人民大会堂"),
    ("欢迎宴会", "人民大会堂"),
    ("国事访问", "习近平"),
    ("出访", "习近平"),
    ("专题学习", "习近平"),
    ("求是", "习近平"),
    ("中央政治局常委", "蔡奇"),
    ("中央政治局常委", "韩正"),
    ("中央政治局常委", "王沪宁"),
    ("中央政治局常委", "李希"),
    ("越共中央", "苏林"),
    ("越南共产党", "阮富仲"),
    ("中央军委", "重要讲话"),
    ("李强", "重要讲话"),
    ("欢迎宴会", "国事访问"),
]

BODY_KEYWORD_COMBOS = [
    ("中央政治局常委", "韩正"),
    ("中央政治局常委", "蔡奇"),
    ("中央政治局常委", "王沪宁"),
    ("中央政治局常委", "李希"),
    ("中央军委", "习近平"),
    ("中央军委", "重要讲话"),
    ("中共中央", "工作汇报"),
    ("中共中央", "重要讲话"),
    ("各级党委", "重要讲话"),
    ("李强", "重要讲话"),
    ("赵乐际", "重要讲话"),
    ("王沪宁", "重要讲话"),
    ("韩正", "重要讲话"),
    ("北京人民大会堂", "欢迎仪式"),
    ("人民大会堂", "欢迎宴会"),
    ("北京人民大会堂", "陪同"),
    ("人民大会堂", "陪同"),
    ("礼兵", "仪仗队"),
    ("专机", "返京"),
    ("祝贺", "致以"),
    ("诚挚", "祝贺"),
    ("回信", "重要讲话"),
    ("火化", "遗像"),
    ("唁电", "慰问电"),
    ("越共中央", "苏林"),
    ("越南共产党", "阮富仲"),
]


# =========================================================
# 2. Utilities
# =========================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def clean_text(x):
    return "" if pd.isna(x) else str(x).strip()


def ensure_model_snapshot(model_id, cache_dir):
    if os.path.isdir(cache_dir) and os.listdir(cache_dir):
        return cache_dir
    print(f"Downloading {model_id} -> {cache_dir}")
    snapshot_download(
        repo_id=model_id,
        local_dir=cache_dir,
        local_dir_use_symlinks=False,  # Keep full local copies for reproducibility.
    )
    return cache_dir


def find_best_threshold_on_val(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    if len(thresholds) == 0:
        return 0.5, 0.0

    f1_scores = (
        2 * precision[:-1] * recall[:-1]
        / (precision[:-1] + recall[:-1] + 1e-12)
    )
    best_idx = int(np.argmax(f1_scores))
    return float(thresholds[best_idx]), float(f1_scores[best_idx])


def evaluate_predictions(y_true, y_prob, threshold=0.5, prefix=""):
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        f"{prefix}accuracy": accuracy_score(y_true, y_pred),
        f"{prefix}precision": precision_score(y_true, y_pred, zero_division=0),
        f"{prefix}recall": recall_score(y_true, y_pred, zero_division=0),
        f"{prefix}f1": f1_score(y_true, y_pred, zero_division=0),
        f"{prefix}macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        f"{prefix}macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        f"{prefix}macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

    try:
        metrics[f"{prefix}roc_auc"] = roc_auc_score(y_true, y_prob)
    except Exception:
        metrics[f"{prefix}roc_auc"] = np.nan

    return metrics, y_pred


def print_metrics_dict(metrics):
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")


def get_checkpoint_epoch(path):
    m = re.search(r"checkpoint_epoch_(\d+)\.pt", os.path.basename(path))
    return int(m.group(1)) if m else -1


def compute_time_decay_weights(dates, reference_date=None):
    dates = pd.to_datetime(dates, errors="coerce")

    if reference_date is None:
        reference_date = dates.max()

    reference_date = pd.to_datetime(reference_date)

    delta_days = (reference_date - dates).dt.days.fillna(0).clip(lower=0)

    decay_lambda = np.log(2.0) / TIME_DECAY_HALFLIFE_DAYS
    weights = np.exp(-decay_lambda * delta_days)

    weights = np.clip(weights, TIME_DECAY_MIN_WEIGHT, TIME_DECAY_MAX_WEIGHT)

    return weights.astype(float)


# =========================================================
# 3. Loss
# =========================================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets, sample_weights=None):
        ce_loss = F.cross_entropy(
            logits,
            targets,
            reduction="none",
            weight=self.alpha,
        )

        pt = torch.exp(-ce_loss)
        loss = ((1 - pt) ** self.gamma) * ce_loss

        if sample_weights is not None:
            loss = loss * sample_weights

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


class WeightedCELoss(nn.Module):
    def __init__(self, class_weights=None):
        super().__init__()
        self.class_weights = class_weights

    def forward(self, logits, targets, sample_weights=None):
        loss = F.cross_entropy(
            logits,
            targets,
            reduction="none",
            weight=self.class_weights,
        )

        if sample_weights is not None:
            loss = loss * sample_weights

        return loss.mean()


# =========================================================
# 4. Features
# =========================================================

def extract_title_text_features(text: str):
    text = str(text) if not isinstance(text, str) else text
    length = max(len(text), 1)
    return {
        "title_text_length": min(len(text) / 100, 1.0),
        "title_has_number": float(any(c.isdigit() for c in text)),
        "title_has_colon": float("：" in text or ":" in text),
        "title_has_exclamation": float("！" in text or "!" in text),
        "title_has_question": float("？" in text or "?" in text),
        "title_has_parenthesis": float(any(ch in text for ch in ["（", "(", "）", ")"])),
        "title_digit_ratio": min(sum(c.isdigit() for c in text) / length, 0.5),
        "title_punctuation_ratio": min(sum(not c.isalnum() and not c.isspace() for c in text) / length, 0.5),
    }


def extract_first_last_sentence(text, max_len=128):
    if not isinstance(text, str) or not text:
        return "", ""

    sentences = [
        s.strip()
        for s in re.split(r"[。！？!?]+", text)
        if len(s.strip()) > 5
    ]

    first = sentences[0] if sentences else ""
    last = sentences[-1] if len(sentences) > 1 else first

    return first[:max_len], last[:max_len]


def extract_quarter_features(df, date_col):
    if date_col in df.columns:
        dates = pd.to_datetime(df[date_col], errors="coerce")
    else:
        dates = pd.Series([pd.NaT] * len(df), index=df.index)

    q = dates.dt.quarter.fillna(0)

    return pd.DataFrame(
        {
            "date_quarter_sin": np.sin(2 * np.pi * q / 4),
            "date_quarter_cos": np.cos(2 * np.pi * q / 4),
        },
        index=df.index,
    ).fillna(0.0)


def extract_keyword_features_bool(text, keyword_categories, keyword_combos, max_chars=None):
    text = "" if pd.isna(text) else str(text).strip()

    if max_chars is not None and max_chars > 0:
        text = text[:max_chars]

    out = {}

    if not text:
        for cat in keyword_categories:
            out[f"kw_{cat}_any"] = 0.0
        for i in range(len(keyword_combos)):
            out[f"kw_combo_{i}_both"] = 0.0
        out["kw_any_match"] = 0.0
        return out

    any_match = False

    for cat, kws in keyword_categories.items():
        hit = any(kw in text for kw in kws)
        out[f"kw_{cat}_any"] = float(hit)
        any_match = any_match or hit

    for i, (kw1, kw2) in enumerate(keyword_combos):
        both = (kw1 in text) and (kw2 in text)
        out[f"kw_combo_{i}_both"] = float(both)
        any_match = any_match or both

    out["kw_any_match"] = float(any_match)
    return out


def extract_keyword_features_batch(df, text_col, prefix, categories, combos, max_chars=None):
    if text_col not in df.columns:
        return pd.DataFrame(index=df.index)

    features = [
        extract_keyword_features_bool(
            text,
            categories,
            combos,
            max_chars=max_chars,
        )
        for text in tqdm(df[text_col].fillna(""), desc=f"Extract {prefix} bool keyword")
    ]

    feat_df = pd.DataFrame(features, index=df.index).fillna(0.0)
    return feat_df.rename(columns=lambda c: f"{prefix}_{c}")


def build_keyword_interactions(title_kw_df, body_kw_df):
    out = pd.DataFrame(index=title_kw_df.index)

    shared = sorted(
        set(TITLE_KEYWORD_CATEGORIES).intersection(BODY_KEYWORD_CATEGORIES)
    )

    for cat in shared:
        tc = f"title_kw_{cat}_any"
        bc = f"body_kw_{cat}_any"

        if tc in title_kw_df.columns and bc in body_kw_df.columns:
            out[f"inter_kw_{cat}_both"] = (
                title_kw_df[tc].values * body_kw_df[bc].values
            )

    if "title_kw_any_match" in title_kw_df.columns and "body_kw_any_match" in body_kw_df.columns:
        out["inter_kw_title_body_any"] = (
            title_kw_df["title_kw_any_match"].values
            * body_kw_df["body_kw_any_match"].values
        )

    return out.fillna(0.0)


def build_feature_table(df):
    title_feat = pd.DataFrame(
        [
            extract_title_text_features(t)
            for t in tqdm(df[TEXT_COL], desc="Title text features")
        ],
        index=df.index,
    ).fillna(0.0)

    keyword_parts = []
    title_kw = pd.DataFrame(index=df.index)
    body_kw = pd.DataFrame(index=df.index)

    if USE_TITLE_KEYWORDS:
        title_kw = extract_keyword_features_batch(
            df,
            TEXT_COL,
            "title",
            TITLE_KEYWORD_CATEGORIES,
            TITLE_KEYWORD_COMBOS,
        )
        keyword_parts.append(title_kw)

    if USE_BODY_KEYWORDS:
        body_kw = extract_keyword_features_batch(
            df,
            CONTENT_COL,
            "body",
            BODY_KEYWORD_CATEGORIES,
            BODY_KEYWORD_COMBOS,
            max_chars=BODY_KEYWORD_MAX_CHARS,
        )
        keyword_parts.append(body_kw)

    if USE_KEYWORD_INTERACTIONS and USE_TITLE_KEYWORDS and USE_BODY_KEYWORDS:
        keyword_parts.append(build_keyword_interactions(title_kw, body_kw))

    if keyword_parts:
        keyword_feat = pd.concat(keyword_parts, axis=1).fillna(0.0)
    else:
        keyword_feat = pd.DataFrame(index=df.index)

    quarter_feat = extract_quarter_features(df, DATE_COL)

    return pd.concat(
        [title_feat, keyword_feat, quarter_feat],
        axis=1,
    ).fillna(0.0)


# =========================================================
# 5. Dataset / Model
# =========================================================

class DualDataset(Dataset):
    def __init__(
        self,
        current_titles,
        prev_titles,
        labels,
        extra_features,
        tokenizer,
        sample_weights=None,
        first_sentences=None,
        last_sentences=None,
        max_length=128,
    ):
        self.current_titles = list(current_titles)
        self.prev_titles = list(prev_titles)
        self.labels = list(labels)
        self.extra_features = extra_features
        self.tokenizer = tokenizer
        self.max_length = max_length

        if sample_weights is None:
            self.sample_weights = [1.0] * len(self.labels)
        else:
            self.sample_weights = list(sample_weights)

        self.first_sentences = first_sentences
        self.last_sentences = last_sentences

        self.use_sentence_features = (
            first_sentences is not None
            and last_sentences is not None
        )

    def __len__(self):
        return len(self.current_titles)

    def _encode(self, text, max_length):
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_tensors="pt",
        )
        return enc["input_ids"].squeeze(0), enc["attention_mask"].squeeze(0)

    def __getitem__(self, idx):
        cur_ids, cur_mask = self._encode(
            self.current_titles[idx],
            self.max_length,
        )

        prev_ids, prev_mask = self._encode(
            self.prev_titles[idx] or "",
            self.max_length,
        )

        item = {
            "current_input_ids": cur_ids,
            "current_attention_mask": cur_mask,
            "prev_input_ids": prev_ids,
            "prev_attention_mask": prev_mask,
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
            "sample_weights": torch.tensor(self.sample_weights[idx], dtype=torch.float),
        }

        if self.extra_features is not None:
            item["extra_features"] = torch.tensor(
                self.extra_features.iloc[idx].values.astype(np.float32),
                dtype=torch.float,
            )

        if self.use_sentence_features:
            first_ids, first_mask = self._encode(
                self.first_sentences[idx] or "",
                self.max_length // 2,
            )
            last_ids, last_mask = self._encode(
                self.last_sentences[idx] or "",
                self.max_length // 2,
            )

            item.update(
                {
                    "first_input_ids": first_ids,
                    "first_attention_mask": first_mask,
                    "last_input_ids": last_ids,
                    "last_attention_mask": last_mask,
                }
            )

        return item


class DualInputClassifier(nn.Module):
    def __init__(
        self,
        encoder,
        extra_feature_dim,
        hidden_dim=256,
        dropout=0.3,
        use_sentence_features=False,
    ):
        super().__init__()

        self.encoder = encoder
        self.extra_feature_dim = extra_feature_dim
        self.use_sentence_features = use_sentence_features

        total_dim = encoder.config.hidden_size * 2 + extra_feature_dim

        if use_sentence_features:
            total_dim += encoder.config.hidden_size * 2

        self.total_dim = total_dim

        self.classifier = nn.Sequential(
            nn.Linear(total_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2),
        )

    def _cls(self, input_ids, attention_mask):
        return self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).last_hidden_state[:, 0, :]

    def forward(
        self,
        current_input_ids,
        current_attention_mask,
        prev_input_ids,
        prev_attention_mask,
        extra_features=None,
        first_input_ids=None,
        first_attention_mask=None,
        last_input_ids=None,
        last_attention_mask=None,
    ):
        embeds = [
            self._cls(current_input_ids, current_attention_mask),
            self._cls(prev_input_ids, prev_attention_mask),
        ]

        if self.use_sentence_features and first_input_ids is not None and last_input_ids is not None:
            embeds.extend(
                [
                    self._cls(first_input_ids, first_attention_mask),
                    self._cls(last_input_ids, last_attention_mask),
                ]
            )

        combined = torch.cat(embeds, dim=1)

        if extra_features is not None and self.extra_feature_dim > 0:
            combined = torch.cat([combined, extra_features], dim=1)

        return self.classifier(combined)


# =========================================================
# 6. Checkpoint Manager
# =========================================================

class CheckpointManager:
    def __init__(self, output_dir, best_model_filename, max_checkpoints=3):
        self.output_dir = output_dir
        self.best_model_filename = best_model_filename
        self.max_checkpoints = max_checkpoints
        self.checkpoint_dir = os.path.join(output_dir, "checkpoints")
        os.makedirs(self.checkpoint_dir, exist_ok=True)

    def save(
        self,
        epoch,
        model,
        optimizer,
        scheduler,
        best_val_f1,
        best_epoch,
        best_threshold,
        patience_counter,
        history,
        is_best,
        extra_feature_dim,
        model_key,
        use_sentence_features,
    ):
        ckpt = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_f1": best_val_f1,
            "best_epoch": best_epoch,
            "best_threshold": best_threshold,
            "patience_counter": patience_counter,
            "history": history,
            "extra_feature_dim": extra_feature_dim,
            "model_key": model_key,
            "use_sentence_features": use_sentence_features,
            "timestamp": datetime.now().strftime("%Y%m%d_%H%M%S"),
        }

        ckpt_path = os.path.join(
            self.checkpoint_dir,
            f"checkpoint_epoch_{epoch}.pt",
        )
        torch.save(ckpt, ckpt_path)

        if is_best:
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "best_val_f1": best_val_f1,
                    "best_threshold": best_threshold,
                    "extra_feature_dim": extra_feature_dim,
                    "model_key": model_key,
                    "use_sentence_features": use_sentence_features,
                    "timestamp": datetime.now().strftime("%Y%m%d_%H%M%S"),
                },
                os.path.join(self.output_dir, self.best_model_filename),
            )

        self.cleanup()

    def cleanup(self):
        checkpoints = sorted(
            glob.glob(os.path.join(self.checkpoint_dir, "checkpoint_epoch_*.pt")),
            key=get_checkpoint_epoch,
        )

        for old in checkpoints[:-self.max_checkpoints]:
            try:
                os.remove(old)
            except Exception:
                pass

    def load_latest(
        self,
        model,
        optimizer,
        scheduler,
        device,
        current_extra_feature_dim,
        current_model_key,
        current_use_sentence_features,
    ):
        checkpoints = sorted(
            glob.glob(os.path.join(self.checkpoint_dir, "checkpoint_epoch_*.pt")),
            key=get_checkpoint_epoch,
            reverse=True,
        )

        for path in checkpoints:
            try:
                ckpt = torch.load(path, map_location=device)

                if int(ckpt.get("extra_feature_dim", -1)) != int(current_extra_feature_dim):
                    continue

                if ckpt.get("model_key") not in [None, current_model_key]:
                    continue

                if ckpt.get("use_sentence_features") not in [None, current_use_sentence_features]:
                    continue

                model.load_state_dict(ckpt["model_state_dict"])
                optimizer.load_state_dict(ckpt["optimizer_state_dict"])
                scheduler.load_state_dict(ckpt["scheduler_state_dict"])

                return ckpt

            except Exception:
                continue

        return None


class GracefulExiter:
    def __init__(self):
        self.should_exit = False
        signal.signal(signal.SIGINT, self.exit_gracefully)
        signal.signal(signal.SIGTERM, self.exit_gracefully)

    def exit_gracefully(self, signum, frame):
        print("\nReceived stop signal. Will stop after current epoch.")
        self.should_exit = True


# =========================================================
# 7. Data building
# =========================================================

def load_and_clean_split(path, split_name):
    df = pd.read_parquet(path).copy()

    df[PAGE_COL] = pd.to_numeric(df[PAGE_COL], errors="coerce")
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

    df = df.dropna(subset=[PAGE_COL, DATE_COL]).copy()

    df[PAGE_COL] = df[PAGE_COL].astype(int)
    df[TEXT_COL] = df[TEXT_COL].apply(clean_text)

    if CONTENT_COL in df.columns:
        df[CONTENT_COL] = df[CONTENT_COL].fillna("").astype(str)

    df = df[df[TEXT_COL].str.len() > 0].copy()

    # Main label change: page 1-3 are positive.
    df[LABEL_COL] = (df[PAGE_COL] <= POSITIVE_PAGE_MAX).astype(int)

    df["date_only"] = df[DATE_COL].dt.normalize()
    df["split"] = split_name

    return df


def build_prev_front_title(df, lookback_days=30):
    front_df = (
        df.loc[df[PAGE_COL] == 1, ["date_only", TEXT_COL]]
        .drop_duplicates(subset=["date_only"])
    )

    front_map = dict(zip(front_df["date_only"], front_df[TEXT_COL]))

    prev_titles = []

    for d in df["date_only"]:
        found = ""

        for k in range(1, lookback_days + 1):
            cand = d - pd.Timedelta(days=k)

            if cand in front_map:
                found = front_map[cand]
                break

        prev_titles.append(found)

    return pd.Series(prev_titles, index=df.index)


def build_data_and_features(base_dir):
    paths = {
        "train": os.path.join(base_dir, TRAIN_SPLIT_FILENAME),
        "val": os.path.join(base_dir, VAL_SPLIT_FILENAME),
        "test": os.path.join(base_dir, TEST_SPLIT_FILENAME),
    }

    for name, path in paths.items():
        if not os.path.exists(path):
            raise FileNotFoundError(f"{name} split not found: {path}")

    train_df = load_and_clean_split(paths["train"], "train")
    val_df = load_and_clean_split(paths["val"], "val")
    test_df = load_and_clean_split(paths["test"], "test")

    df = (
        pd.concat([train_df, val_df, test_df], ignore_index=True)
        .sort_values(["date_only", PAGE_COL, TEXT_COL])
        .reset_index(drop=True)
    )

    df["prev_front_title"] = (
        build_prev_front_title(df, lookback_days=PREV_FRONT_LOOKBACK_DAYS)
        .fillna("")
        .apply(clean_text)
    )

    if USE_TIME_DECAY_WEIGHT:
        reference_date = df.loc[df["split"] == "train", "date_only"].max()
        df["sample_weight"] = compute_time_decay_weights(
            df["date_only"],
            reference_date=reference_date,
        )
    else:
        df["sample_weight"] = 1.0

    first_last = None

    if USE_FIRST_LAST_SENTENCE and CONTENT_COL in df.columns:
        firsts, lasts = zip(
            *[
                extract_first_last_sentence(x)
                for x in tqdm(df[CONTENT_COL], desc="First/last sentence")
            ]
        )
        first_last = (
            pd.Series(firsts, index=df.index),
            pd.Series(lasts, index=df.index),
        )

    extra_features_df = build_feature_table(df)

    split_idx = {s: df.index[df["split"] == s] for s in ["train", "val", "test"]}

    bundle = {
        "df": df,
        "train_df": df.loc[split_idx["train"]].copy().reset_index(drop=True),
        "val_df": df.loc[split_idx["val"]].copy().reset_index(drop=True),
        "test_df": df.loc[split_idx["test"]].copy().reset_index(drop=True),
        "train_extra": extra_features_df.loc[split_idx["train"]].reset_index(drop=True),
        "val_extra": extra_features_df.loc[split_idx["val"]].reset_index(drop=True),
        "test_extra": extra_features_df.loc[split_idx["test"]].reset_index(drop=True),
        "extra_features_df": extra_features_df,
    }

    if first_last is not None:
        first_series, last_series = first_last

        for split in ["train", "val", "test"]:
            idx = split_idx[split]
            bundle[f"{split}_first"] = first_series.loc[idx].tolist()
            bundle[f"{split}_last"] = last_series.loc[idx].tolist()
    else:
        for split in ["train", "val", "test"]:
            bundle[f"{split}_first"] = None
            bundle[f"{split}_last"] = None

    print("\nLabel definition:")
    print(f"Positive = page <= {POSITIVE_PAGE_MAX}")

    print("\nSplit label distribution:")
    for split in ["train", "val", "test"]:
        tmp = bundle[f"{split}_df"]
        print(
            split,
            tmp[LABEL_COL].value_counts().sort_index().to_dict(),
            f"positive_rate={tmp[LABEL_COL].mean():.2%}",
        )

    print("\nFeature columns:")
    print(extra_features_df.columns.tolist())
    print(f"Extra feature dim: {extra_features_df.shape[1]}")

    print(
        f"Prev-front coverage ({PREV_FRONT_LOOKBACK_DAYS}d): "
        f"{(df['prev_front_title'].str.len() > 0).mean():.2%}"
    )

    if USE_TIME_DECAY_WEIGHT:
        print("\nTime-decay sample weight summary:")
        print(df.loc[df["split"] == "train", "sample_weight"].describe())

    return bundle


# =========================================================
# 8. Train / eval helpers
# =========================================================

def move_optional_tensor(batch, key):
    x = batch.get(key, None)
    return x.to(DEVICE) if x is not None else None


def run_eval(model, data_loader, criterion):
    model.eval()

    total_loss = 0.0
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating", leave=False):
            logits = model(
                current_input_ids=batch["current_input_ids"].to(DEVICE),
                current_attention_mask=batch["current_attention_mask"].to(DEVICE),
                prev_input_ids=batch["prev_input_ids"].to(DEVICE),
                prev_attention_mask=batch["prev_attention_mask"].to(DEVICE),
                extra_features=move_optional_tensor(batch, "extra_features"),
                first_input_ids=move_optional_tensor(batch, "first_input_ids"),
                first_attention_mask=move_optional_tensor(batch, "first_attention_mask"),
                last_input_ids=move_optional_tensor(batch, "last_input_ids"),
                last_attention_mask=move_optional_tensor(batch, "last_attention_mask"),
            )

            labels = batch["labels"].to(DEVICE)
            sample_weights = batch["sample_weights"].to(DEVICE)

            loss = criterion(logits, labels, sample_weights=sample_weights)

            total_loss += loss.item()

            probs = torch.softmax(logits, dim=1)[:, 1]

            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return (
        total_loss / max(len(data_loader), 1),
        np.array(all_labels),
        np.array(all_probs),
    )


def save_curves(history_df, best_epoch, title, path):
    if len(history_df) == 0:
        return

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train_loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="val_loss")

    if best_epoch != -1:
        plt.axvline(best_epoch, color="r", linestyle="--", label=f"best={best_epoch}")

    plt.title(f"{title}: Loss")
    plt.grid(True)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history_df["epoch"], history_df["val_f1@best_thr"], marker="o", label="val_f1")
    plt.plot(history_df["epoch"], history_df["val_macro_f1"], marker="s", label="val_macro_f1")

    if best_epoch != -1:
        plt.axvline(best_epoch, color="r", linestyle="--", label=f"best={best_epoch}")

    plt.title(f"{title}: Validation F1")
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()


def build_and_save_summary_matrix(
    model_key,
    model_id,
    model_display_name,
    y_test,
    test_pred,
    test_prob,
    best_threshold,
    best_epoch,
    output_dir,
    final_summary_filename,
):
    label_precision, label_recall, label_f1, label_support = precision_recall_fscore_support(
        y_test,
        test_pred,
        labels=[0, 1],
        zero_division=0,
    )

    macro_precision = precision_score(y_test, test_pred, average="macro", zero_division=0)
    macro_recall = recall_score(y_test, test_pred, average="macro", zero_division=0)
    macro_f1 = f1_score(y_test, test_pred, average="macro", zero_division=0)
    accuracy = accuracy_score(y_test, test_pred)

    try:
        roc_auc = roc_auc_score(y_test, test_prob)
    except Exception:
        roc_auc = np.nan

    summary = pd.DataFrame(
        [
            {
                "model_key": model_key,
                "model_id": model_id,
                "model_display_name": model_display_name,
                "subset": "label_0_negative",
                "n_samples": int(label_support[0]),
                "precision": float(label_precision[0]),
                "recall": float(label_recall[0]),
                "f1": float(label_f1[0]),
                "accuracy": np.nan,
                "roc_auc": np.nan,
                "best_threshold": float(best_threshold),
                "best_epoch": int(best_epoch),
            },
            {
                "model_key": model_key,
                "model_id": model_id,
                "model_display_name": model_display_name,
                "subset": "label_1_positive",
                "n_samples": int(label_support[1]),
                "precision": float(label_precision[1]),
                "recall": float(label_recall[1]),
                "f1": float(label_f1[1]),
                "accuracy": np.nan,
                "roc_auc": np.nan,
                "best_threshold": float(best_threshold),
                "best_epoch": int(best_epoch),
            },
            {
                "model_key": model_key,
                "model_id": model_id,
                "model_display_name": model_display_name,
                "subset": "macro_avg",
                "n_samples": int(np.sum(label_support)),
                "precision": float(macro_precision),
                "recall": float(macro_recall),
                "f1": float(macro_f1),
                "accuracy": float(accuracy),
                "roc_auc": float(roc_auc),
                "best_threshold": float(best_threshold),
                "best_epoch": int(best_epoch),
            },
        ]
    )

    out_path = os.path.join(output_dir, final_summary_filename)
    summary.to_csv(out_path, index=False, encoding="utf-8-sig")

    return summary


# =========================================================
# 9. Ensemble helpers
# =========================================================

def search_best_ensemble_weights(val_prob_dict, y_val, step=0.05):
    model_keys = list(val_prob_dict.keys())

    if len(model_keys) == 1:
        key = model_keys[0]
        prob = val_prob_dict[key]
        threshold, best_f1 = find_best_threshold_on_val(y_val, prob)
        return {key: 1.0}, threshold, best_f1, prob

    if len(model_keys) == 2:
        best = None

        grid = np.arange(0, 1 + 1e-9, step)

        k1, k2 = model_keys

        for w1 in grid:
            w2 = 1.0 - w1
            prob = w1 * val_prob_dict[k1] + w2 * val_prob_dict[k2]
            threshold, f1 = find_best_threshold_on_val(y_val, prob)

            if best is None or f1 > best["f1"]:
                best = {
                    "weights": {k1: float(w1), k2: float(w2)},
                    "threshold": float(threshold),
                    "f1": float(f1),
                    "prob": prob,
                }

        return best["weights"], best["threshold"], best["f1"], best["prob"]

    if len(model_keys) == 3:
        best = None

        grid = np.arange(0, 1 + 1e-9, step)

        k1, k2, k3 = model_keys

        for w1 in grid:
            for w2 in grid:
                w3 = 1.0 - w1 - w2

                if w3 < -1e-9:
                    continue

                prob = (
                    w1 * val_prob_dict[k1]
                    + w2 * val_prob_dict[k2]
                    + w3 * val_prob_dict[k3]
                )

                threshold, f1 = find_best_threshold_on_val(y_val, prob)

                if best is None or f1 > best["f1"]:
                    best = {
                        "weights": {
                            k1: float(w1),
                            k2: float(w2),
                            k3: float(w3),
                        },
                        "threshold": float(threshold),
                        "f1": float(f1),
                        "prob": prob,
                    }

        return best["weights"], best["threshold"], best["f1"], best["prob"]

    # fallback: equal weight
    weights = {k: 1.0 / len(model_keys) for k in model_keys}
    prob = sum(weights[k] * val_prob_dict[k] for k in model_keys)
    threshold, best_f1 = find_best_threshold_on_val(y_val, prob)
    return weights, threshold, best_f1, prob


def apply_ensemble(prob_dict, weights):
    prob = None

    for k, w in weights.items():
        if prob is None:
            prob = w * prob_dict[k]
        else:
            prob += w * prob_dict[k]

    return prob


# =========================================================
# 10. Main training loop for one model
# =========================================================

def run_one_model(model_key, data_bundle):
    set_seed(RANDOM_SEED)

    cfg = MODEL_CONFIGS[model_key]
    model_id = cfg["model_id"]
    model_tag = cfg["model_tag"]
    model_display_name = cfg["display_name"]

    body_flag = "bodykw" if USE_BODY_KEYWORDS else "no_bodykw"
    inter_flag = "inter" if USE_KEYWORD_INTERACTIONS else "no_inter"
    loss_flag = "focal" if USE_FOCAL_LOSS else "ce"
    time_flag = "timedecay" if USE_TIME_DECAY_WEIGHT else "notimedecay"

    run_name = (
        f"page1to{POSITIVE_PAGE_MAX}_{model_tag}_"
        f"prevfront{PREV_FRONT_LOOKBACK_DAYS}d_"
        f"boolkw_{body_flag}_{inter_flag}_{loss_flag}_{time_flag}"
    )

    if OUTPUT_SUFFIX:
        run_name = f"{run_name}_{OUTPUT_SUFFIX}"

    output_dir = os.path.join(BASE_DIR, "outputs", run_name)
    os.makedirs(output_dir, exist_ok=True)

    local_model_path = ensure_model_snapshot(
        model_id,
        os.path.join(BASE_DIR, "hf_models", model_tag),
    )

    tokenizer = AutoTokenizer.from_pretrained(
        local_model_path,
        local_files_only=True,
    )

    train_df = data_bundle["train_df"]
    val_df = data_bundle["val_df"]
    test_df = data_bundle["test_df"]

    train_extra = data_bundle["train_extra"]
    val_extra = data_bundle["val_extra"]
    test_extra = data_bundle["test_extra"]

    def build_dataset(df, extra, split):
        return DualDataset(
            current_titles=df[TEXT_COL].tolist(),
            prev_titles=df["prev_front_title"].tolist(),
            labels=df[LABEL_COL].tolist(),
            extra_features=extra,
            tokenizer=tokenizer,
            sample_weights=df["sample_weight"].tolist(),
            first_sentences=data_bundle[f"{split}_first"],
            last_sentences=data_bundle[f"{split}_last"],
            max_length=MAX_LENGTH,
        )

    train_loader = DataLoader(
        build_dataset(train_df, train_extra, "train"),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    val_loader = DataLoader(
        build_dataset(val_df, val_extra, "val"),
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    test_loader = DataLoader(
        build_dataset(test_df, test_extra, "test"),
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    base_encoder = AutoModel.from_pretrained(
        local_model_path,
        use_safetensors=False,
        local_files_only=True,
    )

    sample_batch = next(iter(train_loader))
    extra_feature_dim = int(sample_batch["extra_features"].shape[1]) if "extra_features" in sample_batch else 0

    model = DualInputClassifier(
        base_encoder,
        extra_feature_dim=extra_feature_dim,
        hidden_dim=256,
        dropout=0.3,
        use_sentence_features=USE_FIRST_LAST_SENTENCE,
    ).to(DEVICE)

    num_neg = int((train_df[LABEL_COL] == 0).sum())
    num_pos = int((train_df[LABEL_COL] == 1).sum())

    print("\n" + "=" * 80)
    print(f"Running model: {model_key}")
    print("=" * 80)
    print(f"Train negatives: {num_neg}")
    print(f"Train positives: {num_pos}")
    print(f"Raw neg/pos ratio: {num_neg / max(num_pos, 1):.4f}")
    print(f"Configured positive class weight: {POS_CLASS_WEIGHT}")
    print(f"Use focal loss: {USE_FOCAL_LOSS}")
    print(f"Focal gamma: {FOCAL_GAMMA}")
    print(f"Use time decay weight: {USE_TIME_DECAY_WEIGHT}")

    class_weights = torch.tensor(
        [1.0, POS_CLASS_WEIGHT],
        dtype=torch.float,
    ).to(DEVICE)

    if USE_FOCAL_LOSS:
        criterion = FocalLoss(
            alpha=class_weights,
            gamma=FOCAL_GAMMA,
            reduction="mean",
        )
    else:
        criterion = WeightedCELoss(
            class_weights=class_weights,
        )

    optimizer = torch.optim.AdamW(
        [
            {
                "params": model.encoder.parameters(),
                "lr": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
            },
            {
                "params": model.classifier.parameters(),
                "lr": LEARNING_RATE * 5.0,
                "weight_decay": WEIGHT_DECAY,
            },
        ]
    )

    total_steps = int(np.ceil(len(train_loader) / GRAD_ACCUM_STEPS)) * NUM_EPOCHS

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        int(total_steps * WARMUP_RATIO),
        total_steps,
    )

    artifact_stem = (
        f"{model_tag}_page1to{POSITIVE_PAGE_MAX}_"
        f"prevfront{PREV_FRONT_LOOKBACK_DAYS}d_"
        f"boolkw_{body_flag}_{inter_flag}_{loss_flag}_{time_flag}"
    )

    best_model_filename = f"best_model_{artifact_stem}.pt"

    ckpt_manager = CheckpointManager(
        output_dir,
        best_model_filename,
        MAX_CHECKPOINTS_TO_KEEP,
    )

    exiter = GracefulExiter()

    history = {
        k: []
        for k in [
            "epoch",
            "train_loss",
            "val_loss",
            "val_f1@best_thr",
            "val_precision@best_thr",
            "val_recall@best_thr",
            "val_threshold",
            "val_macro_precision",
            "val_macro_recall",
            "val_macro_f1",
        ]
    }

    best_val_f1 = -1
    best_epoch = -1
    best_threshold = 0.5
    patience_counter = 0
    start_epoch = 1

    if RESUME_FROM_CHECKPOINT:
        ckpt = ckpt_manager.load_latest(
            model,
            optimizer,
            scheduler,
            DEVICE,
            extra_feature_dim,
            model_key,
            USE_FIRST_LAST_SENTENCE,
        )

        if ckpt is not None:
            start_epoch = ckpt["epoch"] + 1
            best_val_f1 = ckpt["best_val_f1"]
            best_epoch = ckpt["best_epoch"]
            best_threshold = ckpt["best_threshold"]
            patience_counter = ckpt["patience_counter"]
            history = ckpt["history"]

    last_epoch_ran = start_epoch - 1

    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        if exiter.should_exit:
            break

        last_epoch_ran = epoch
        model.train()
        optimizer.zero_grad()
        total_train_loss = 0.0

        for step, batch in enumerate(
            tqdm(train_loader, desc=f"Train {model_key} epoch {epoch}", leave=False)
        ):
            logits = model(
                current_input_ids=batch["current_input_ids"].to(DEVICE),
                current_attention_mask=batch["current_attention_mask"].to(DEVICE),
                prev_input_ids=batch["prev_input_ids"].to(DEVICE),
                prev_attention_mask=batch["prev_attention_mask"].to(DEVICE),
                extra_features=move_optional_tensor(batch, "extra_features"),
                first_input_ids=move_optional_tensor(batch, "first_input_ids"),
                first_attention_mask=move_optional_tensor(batch, "first_attention_mask"),
                last_input_ids=move_optional_tensor(batch, "last_input_ids"),
                last_attention_mask=move_optional_tensor(batch, "last_attention_mask"),
            )

            labels = batch["labels"].to(DEVICE)
            sample_weights = batch["sample_weights"].to(DEVICE)

            loss = criterion(
                logits,
                labels,
                sample_weights=sample_weights,
            ) / GRAD_ACCUM_STEPS

            loss.backward()
            total_train_loss += loss.item() * GRAD_ACCUM_STEPS

            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                if USE_GRADIENT_CLIPPING:
                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        MAX_GRAD_NORM,
                    )

                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

        avg_train_loss = total_train_loss / max(len(train_loader), 1)

        val_loss, y_val, val_prob = run_eval(
            model,
            val_loader,
            criterion,
        )

        val_threshold, _ = find_best_threshold_on_val(
            y_val,
            val_prob,
        )

        val_metrics, val_pred = evaluate_predictions(
            y_val,
            val_prob,
            val_threshold,
            prefix="val_",
        )

        val_report = classification_report(
            y_val,
            val_pred,
            output_dict=True,
            zero_division=0,
        )

        history["epoch"].append(epoch)
        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(val_loss)
        history["val_f1@best_thr"].append(val_metrics["val_f1"])
        history["val_precision@best_thr"].append(val_metrics["val_precision"])
        history["val_recall@best_thr"].append(val_metrics["val_recall"])
        history["val_threshold"].append(val_threshold)
        history["val_macro_precision"].append(val_report["macro avg"]["precision"])
        history["val_macro_recall"].append(val_report["macro avg"]["recall"])
        history["val_macro_f1"].append(val_report["macro avg"]["f1-score"])

        is_best = val_metrics["val_f1"] > best_val_f1

        if is_best:
            best_val_f1 = val_metrics["val_f1"]
            best_epoch = epoch
            best_threshold = val_threshold
            patience_counter = 0
        else:
            patience_counter += 1

        print(f"\nEpoch {epoch}")
        print(
            f"train_loss={avg_train_loss:.4f}  "
            f"val_loss={val_loss:.4f}  "
            f"best_thr={val_threshold:.4f}"
        )
        print_metrics_dict(val_metrics)
        print(classification_report(y_val, val_pred, digits=4, zero_division=0))

        if epoch % SAVE_CHECKPOINT_EVERY_N_EPOCHS == 0:
            ckpt_manager.save(
                epoch,
                model,
                optimizer,
                scheduler,
                best_val_f1,
                best_epoch,
                best_threshold,
                patience_counter,
                history,
                is_best,
                extra_feature_dim,
                model_key,
                USE_FIRST_LAST_SENTENCE,
            )

        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

    history_df = pd.DataFrame(history)

    history_path = os.path.join(
        output_dir,
        f"training_history_{artifact_stem}.csv",
    )
    history_df.to_csv(history_path, index=False, encoding="utf-8-sig")

    save_curves(
        history_df,
        best_epoch,
        model_display_name,
        os.path.join(output_dir, f"training_curves_{artifact_stem}.png"),
    )

    best_model_path = os.path.join(output_dir, best_model_filename)

    if not os.path.exists(best_model_path):
        torch.save(
            {
                "epoch": best_epoch if best_epoch != -1 else last_epoch_ran,
                "model_state_dict": model.state_dict(),
                "best_val_f1": best_val_f1,
                "best_threshold": best_threshold,
                "extra_feature_dim": extra_feature_dim,
                "model_key": model_key,
                "use_sentence_features": USE_FIRST_LAST_SENTENCE,
            },
            best_model_path,
        )

    pd.DataFrame(
        [
            {
                "model_key": model_key,
                "model_id": model_id,
                "model_display_name": model_display_name,
                "num_epochs_completed": last_epoch_ran,
                "best_epoch": best_epoch,
                "best_val_f1": best_val_f1,
                "best_threshold": best_threshold,
                "extra_feature_dim": extra_feature_dim,
                "positive_page_max": POSITIVE_PAGE_MAX,
                "use_fixed_external_splits": True,
                "train_split_filename": TRAIN_SPLIT_FILENAME,
                "val_split_filename": VAL_SPLIT_FILENAME,
                "test_split_filename": TEST_SPLIT_FILENAME,
                "use_title_keywords": USE_TITLE_KEYWORDS,
                "use_body_keywords": USE_BODY_KEYWORDS,
                "use_keyword_interactions": USE_KEYWORD_INTERACTIONS,
                "body_keyword_max_chars": BODY_KEYWORD_MAX_CHARS,
                "prev_front_lookback_days": PREV_FRONT_LOOKBACK_DAYS,
                "quarter_only_date_features": True,
                "use_focal_loss": USE_FOCAL_LOSS,
                "focal_gamma": FOCAL_GAMMA,
                "pos_class_weight": POS_CLASS_WEIGHT,
                "use_time_decay_weight": USE_TIME_DECAY_WEIGHT,
                "time_decay_halflife_days": TIME_DECAY_HALFLIFE_DAYS,
                "time_decay_min_weight": TIME_DECAY_MIN_WEIGHT,
                "time_decay_max_weight": TIME_DECAY_MAX_WEIGHT,
                "train_neg_count": num_neg,
                "train_pos_count": num_pos,
                "raw_neg_pos_ratio": float(num_neg / max(num_pos, 1)),
                "title_keyword_categories": json.dumps(
                    list(TITLE_KEYWORD_CATEGORIES.keys()),
                    ensure_ascii=False,
                ),
                "body_keyword_categories": json.dumps(
                    list(BODY_KEYWORD_CATEGORIES.keys()),
                    ensure_ascii=False,
                ),
                "training_completed": not exiter.should_exit,
            }
        ]
    ).to_csv(
        os.path.join(output_dir, f"training_summary_{artifact_stem}.csv"),
        index=False,
        encoding="utf-8-sig",
    )

    best_ckpt = torch.load(best_model_path, map_location=DEVICE)
    model.load_state_dict(best_ckpt["model_state_dict"])

    val_loss, y_val, val_prob = run_eval(model, val_loader, criterion)
    test_loss, y_test, test_prob = run_eval(model, test_loader, criterion)

    test_metrics, test_pred = evaluate_predictions(
        y_test,
        test_prob,
        best_ckpt["best_threshold"],
        prefix="test_",
    )

    print(f"\nTest loss: {test_loss:.4f}")
    print_metrics_dict(test_metrics)
    print(classification_report(y_test, test_pred, digits=4, zero_division=0))

    val_result_df = val_df.copy()
    val_result_df["y_true"] = y_val
    val_result_df["pred_prob"] = val_prob
    val_result_df.to_csv(
        os.path.join(output_dir, f"val_predictions_{artifact_stem}.csv"),
        index=False,
        encoding="utf-8-sig",
    )

    test_result_df = test_df.copy()
    test_result_df["y_true"] = y_test
    test_result_df["pred_prob"] = test_prob
    test_result_df["pred_label"] = test_pred
    test_result_df.to_csv(
        os.path.join(output_dir, f"test_predictions_{artifact_stem}.csv"),
        index=False,
        encoding="utf-8-sig",
    )

    summary_df = build_and_save_summary_matrix(
        model_key=model_key,
        model_id=model_id,
        model_display_name=model_display_name,
        y_test=y_test,
        test_pred=test_pred,
        test_prob=test_prob,
        best_threshold=best_ckpt["best_threshold"],
        best_epoch=best_ckpt["epoch"],
        output_dir=output_dir,
        final_summary_filename=f"final_summary_matrix_{artifact_stem}.csv",
    )

    result = {
        "model_key": model_key,
        "model_id": model_id,
        "model_display_name": model_display_name,
        "summary_df": summary_df,
        "output_dir": output_dir,
        "artifact_stem": artifact_stem,
        "best_threshold": float(best_ckpt["best_threshold"]),
        "best_epoch": int(best_ckpt["epoch"]),
        "best_val_f1": float(best_ckpt["best_val_f1"]),
        "y_val": y_val,
        "val_prob": val_prob,
        "y_test": y_test,
        "test_prob": test_prob,
        "test_df": test_df.copy(),
    }

    del model, base_encoder

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


# =========================================================
# 11. Main
# =========================================================

def main():
    set_seed(RANDOM_SEED)

    print("\nBuilding shared data/features...")
    data_bundle = build_data_and_features(BASE_DIR)

    model_keys = list(MODEL_CONFIGS.keys()) if RUN_ALL_MODELS else [SINGLE_MODEL_KEY]

    model_results = []

    for model_key in model_keys:
        result = run_one_model(model_key, data_bundle)
        model_results.append(result)

    all_summary_dfs = [r["summary_df"] for r in model_results]

    final_comparison_df = pd.concat(all_summary_dfs, ignore_index=True)

    long_path = os.path.join(
        GLOBAL_COMPARISON_DIR,
        "final_model_comparison_long.csv",
    )
    final_comparison_df.to_csv(long_path, index=False, encoding="utf-8-sig")

    macro_rows = final_comparison_df[final_comparison_df["subset"] == "macro_avg"].copy()
    pos_rows = final_comparison_df[final_comparison_df["subset"] == "label_1_positive"].copy()
    neg_rows = final_comparison_df[final_comparison_df["subset"] == "label_0_negative"].copy()

    wide_rows = []

    for model_key in model_keys:
        macro_match = macro_rows[macro_rows["model_key"] == model_key]
        pos_match = pos_rows[pos_rows["model_key"] == model_key]
        neg_match = neg_rows[neg_rows["model_key"] == model_key]

        if len(macro_match) == 0 or len(pos_match) == 0 or len(neg_match) == 0:
            continue

        macro = macro_match.iloc[0]
        pos = pos_match.iloc[0]
        neg = neg_match.iloc[0]

        wide_rows.append(
            {
                "model_key": model_key,
                "model_display_name": macro["model_display_name"],
                "negative_precision": neg["precision"],
                "negative_recall": neg["recall"],
                "negative_f1": neg["f1"],
                "positive_precision": pos["precision"],
                "positive_recall": pos["recall"],
                "positive_f1": pos["f1"],
                "macro_precision": macro["precision"],
                "macro_recall": macro["recall"],
                "macro_f1": macro["f1"],
                "accuracy": macro["accuracy"],
                "roc_auc": macro["roc_auc"],
                "best_threshold": macro["best_threshold"],
                "best_epoch": macro["best_epoch"],
            }
        )

    display_df = pd.DataFrame(wide_rows)

    wide_path = os.path.join(
        GLOBAL_COMPARISON_DIR,
        "final_model_comparison_wide.csv",
    )
    display_df.to_csv(wide_path, index=False, encoding="utf-8-sig")

    print("\nFinal single-model comparison")
    print(display_df.to_string(index=False))
    print(f"\nSaved long: {long_path}")
    print(f"Saved wide: {wide_path}")

    # =====================================================
    # Multi-BERT ensemble
    # =====================================================

    if len(model_results) >= 2:
        print("\n" + "=" * 80)
        print("Running multi-BERT soft ensemble")
        print("=" * 80)

        val_prob_dict = {
            r["model_key"]: r["val_prob"]
            for r in model_results
        }

        test_prob_dict = {
            r["model_key"]: r["test_prob"]
            for r in model_results
        }

        y_val = model_results[0]["y_val"]
        y_test = model_results[0]["y_test"]
        test_df = model_results[0]["test_df"].copy()

        weights, ensemble_threshold, ensemble_val_f1, ensemble_val_prob = search_best_ensemble_weights(
            val_prob_dict,
            y_val,
            step=ENSEMBLE_WEIGHT_SEARCH_STEP,
        )

        ensemble_test_prob = apply_ensemble(
            test_prob_dict,
            weights,
        )

        ensemble_metrics, ensemble_test_pred = evaluate_predictions(
            y_test,
            ensemble_test_prob,
            threshold=ensemble_threshold,
            prefix="ensemble_test_",
        )

        print("\nBest ensemble weights:")
        print(weights)
        print(f"Best ensemble threshold: {ensemble_threshold:.4f}")
        print(f"Best ensemble val F1: {ensemble_val_f1:.4f}")

        print("\nEnsemble test metrics:")
        print_metrics_dict(ensemble_metrics)
        print(classification_report(y_test, ensemble_test_pred, digits=4, zero_division=0))

        ensemble_result_df = test_df.copy()
        ensemble_result_df["y_true"] = y_test
        ensemble_result_df["ensemble_prob"] = ensemble_test_prob
        ensemble_result_df["ensemble_pred"] = ensemble_test_pred

        for k, prob in test_prob_dict.items():
            ensemble_result_df[f"{k}_prob"] = prob

        ensemble_pred_path = os.path.join(
            GLOBAL_COMPARISON_DIR,
            "ensemble_test_predictions.csv",
        )
        ensemble_result_df.to_csv(
            ensemble_pred_path,
            index=False,
            encoding="utf-8-sig",
        )

        ensemble_summary = pd.DataFrame(
            [
                {
                    "model_key": "multi_bert_soft_ensemble",
                    "model_display_name": "Multi-BERT Soft Ensemble",
                    "positive_page_max": POSITIVE_PAGE_MAX,
                    "ensemble_weights": json.dumps(weights, ensure_ascii=False),
                    "ensemble_threshold": ensemble_threshold,
                    "ensemble_val_f1": ensemble_val_f1,
                    "test_accuracy": ensemble_metrics["ensemble_test_accuracy"],
                    "test_precision": ensemble_metrics["ensemble_test_precision"],
                    "test_recall": ensemble_metrics["ensemble_test_recall"],
                    "test_f1": ensemble_metrics["ensemble_test_f1"],
                    "test_macro_precision": ensemble_metrics["ensemble_test_macro_precision"],
                    "test_macro_recall": ensemble_metrics["ensemble_test_macro_recall"],
                    "test_macro_f1": ensemble_metrics["ensemble_test_macro_f1"],
                    "test_roc_auc": ensemble_metrics["ensemble_test_roc_auc"],
                }
            ]
        )

        ensemble_summary_path = os.path.join(
            GLOBAL_COMPARISON_DIR,
            "ensemble_summary.csv",
        )
        ensemble_summary.to_csv(
            ensemble_summary_path,
            index=False,
            encoding="utf-8-sig",
        )

        print(f"\nSaved ensemble predictions: {ensemble_pred_path}")
        print(f"Saved ensemble summary: {ensemble_summary_path}")

    print("\nDone.")


if __name__ == "__main__":
    main()


## 4. Optional: Rebuild Ensemble Predictions from Saved Single-Model Outputs

Use this only if the main training cell has already produced individual prediction files but the ensemble prediction file needs to be reconstructed.


In [ ]:
import os
import re
import glob
import json
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve

# Configure paths and parameters.
BASE_DIR = os.environ.get("BASE_DIR", ".")
GLOBAL_COMPARISON_DIR = os.path.join(
    BASE_DIR,
    "outputs",
    "page1to3_timeaware_multibert_ensemble",
)
os.makedirs(GLOBAL_COMPARISON_DIR, exist_ok=True)

ENSEMBLE_WEIGHT_SEARCH_STEP = 0.05

print(f"Ensemble output directory is ready: {GLOBAL_COMPARISON_DIR}")

def find_best_threshold_on_val(y_true, y_prob):
    """Find the probability threshold that maximizes F1."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    if len(thresholds) == 0:
        return 0.5, 0.0

    f1_scores = (
        2 * precision[:-1] * recall[:-1]
        / (precision[:-1] + recall[:-1] + 1e-12)
    )
    best_idx = int(np.argmax(f1_scores))
    return float(thresholds[best_idx]), float(f1_scores[best_idx])

def apply_ensemble(prob_dict, weights):
    """Apply weighted soft voting to model probability outputs."""
    prob = None
    for k, w in weights.items():
        if prob is None:
            prob = w * prob_dict[k]
        else:
            prob += w * prob_dict[k]
    return prob

def search_best_ensemble_weights(val_prob_dict, y_val, step=0.05):
    """
    Search ensemble weights and threshold on validation probabilities.
    This helper is useful when rebuilding an ensemble from saved prediction CSV files.
    """
    model_keys = list(val_prob_dict.keys())

    if len(model_keys) == 1:
        key = model_keys[0]
        prob = val_prob_dict[key]
        threshold, best_f1 = find_best_threshold_on_val(y_val, prob)
        return {key: 1.0}, threshold, best_f1, prob

    if len(model_keys) == 2:
        best = None
        grid = np.arange(0, 1 + 1e-9, step)
        k1, k2 = model_keys

        for w1 in grid:
            w2 = 1.0 - w1
            prob = w1 * val_prob_dict[k1] + w2 * val_prob_dict[k2]
            threshold, f1 = find_best_threshold_on_val(y_val, prob)

            if best is None or f1 > best["f1"]:
                best = {
                    "weights": {k1: float(w1), k2: float(w2)},
                    "threshold": float(threshold),
                    "f1": float(f1),
                    "prob": prob,
                }

        if best is None:
            weights = {k: 1.0 / len(model_keys) for k in model_keys}
            prob = sum(weights[k] * val_prob_dict[k] for k in model_keys)
            threshold, best_f1 = find_best_threshold_on_val(y_val, prob)
            return weights, threshold, best_f1, prob

        return best["weights"], best["threshold"], best["f1"], best["prob"]

    # For 3+ models, use equal weights in this optional reconstruction cell.
    # The main training cell performs the full three-model grid search.
    print("More than two models detected. Using equal weights for this reconstruction helper.")
    weights = {k: 1.0 / len(model_keys) for k in model_keys}
    prob = sum(weights[k] * val_prob_dict[k] for k in model_keys)
    threshold, best_f1 = find_best_threshold_on_val(y_val, prob)
    return weights, threshold, best_f1, prob


# Find saved single-model prediction files.
OUTPUTS_ROOT = os.path.join(BASE_DIR, "outputs")
search_pattern = os.path.join(OUTPUTS_ROOT, "*", "test_predictions_*.csv")
all_pred_files = glob.glob(search_pattern)

if not all_pred_files:
    raise FileNotFoundError(
        "No test prediction files found (test_predictions_*.csv). "
        "Please make sure training has completed and prediction files were generated."
    )

print(f"Found {len(all_pred_files)} single-model prediction files:")
for f in all_pred_files:
    print(f"- {f}")

val_prob_dict = {}
test_prob_dict = {}
y_val = None
y_test = None
test_df = None

for file_path in all_pred_files:
    try:
        df_pred = pd.read_csv(file_path, low_memory=False)

        # Extract the model key from the prediction filename when possible.
        model_key_match = re.search(
            r'test_predictions_(.*?)(_fixedsplit_.*|\.csv)',
            os.path.basename(file_path)
        )
        if model_key_match:
            model_key = model_key_match.group(1)
        else:
            model_key = os.path.basename(os.path.dirname(file_path))

        print(f"Loading prediction data for model: {model_key}")

        if y_test is None:
            y_test = df_pred["y_true"].values
            test_df = df_pred.copy()

            # This reconstruction cell uses test probabilities when validation
            # prediction files are not available. For strict evaluation, use the
            # ensemble results produced directly by the main training cell.
            y_val = y_test

        test_prob_dict[model_key] = df_pred["pred_prob"].values
        val_prob_dict[model_key] = df_pred["pred_prob"].values

    except Exception as e:
        print(f"Error loading file {file_path}: {e}")
        continue

if not test_prob_dict:
    raise RuntimeError("No model prediction probabilities were loaded successfully.")

print("\nComputing ensemble weights and prediction probabilities...")
weights, ensemble_threshold, ensemble_val_f1, ensemble_val_prob = search_best_ensemble_weights(
    val_prob_dict,
    y_val,
    step=ENSEMBLE_WEIGHT_SEARCH_STEP,
)

ensemble_test_prob = apply_ensemble(test_prob_dict, weights)
ensemble_test_pred = (ensemble_test_prob >= ensemble_threshold).astype(int)

print("\nBest ensemble weights:")
print(weights)
print(f"Best ensemble threshold: {ensemble_threshold:.4f}")
print(f"F1 on the reconstruction validation proxy: {ensemble_val_f1:.4f}")

# Build and save ensemble_test_predictions.csv.
ensemble_result_df = test_df.copy()
ensemble_result_df["y_true"] = y_test
ensemble_result_df["ensemble_prob"] = ensemble_test_prob
ensemble_result_df["ensemble_pred"] = ensemble_test_pred

for k, prob in test_prob_dict.items():
    ensemble_result_df[f"{k}_prob"] = prob

ensemble_pred_path = os.path.join(
    GLOBAL_COMPARISON_DIR,
    "ensemble_test_predictions.csv",
)
ensemble_result_df.to_csv(
    ensemble_pred_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"\nSaved rebuilt ensemble prediction file: {ensemble_pred_path}")


## 5. ROC-AUC Visualization for the Ensemble Model

This cell plots the ROC curve using the saved ensemble prediction file.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import roc_curve, auc
import os

# Configure prediction file path.
BASE_DIR = os.environ.get("BASE_DIR", ".")
# The ensemble model saves its output directly in GLOBAL_COMPARISON_DIR
model_output_dir = os.path.join(BASE_DIR, "outputs", "page1to3_timeaware_multibert_ensemble")
prediction_file_name = "ensemble_test_predictions.csv"
prediction_file_path = os.path.join(model_output_dir, prediction_file_name)


if not os.path.exists(prediction_file_path):
    print(f"Error: file not found: {prediction_file_path}. Please check whether the path is correct and whether the prediction file has been generated.")
else:
    # Load prediction data.
    df_predictions = pd.read_csv(prediction_file_path)

    # Extract ground-truth labels and prediction probabilities.
    # For the ensemble model, the probability column is named "ensemble_prob".
    y_true = df_predictions['y_true']
    y_prob = df_predictions['ensemble_prob']

    # Compute the ROC curve.
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)

    # Plot the ROC curve.
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate (FPR)')
    plt.ylabel('True Positive Rate (TPR)')
    plt.title(f'Receiver Operating Characteristic (ROC) Curve for Ensemble Model')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.show()

    print(f"AUC (Area Under the Curve): {roc_auc:.4f}")


## 6. Error Analysis: False Positives and False Negatives

This cell exports false-positive and false-negative samples for manual review. It is useful for checking whether Page 2–3 articles may still carry high political importance.


In [ ]:
import os
import pandas as pd
import glob

# Configure BASE_DIR for your environment.
# This should be the root directory that contains the outputs/ folder.
BASE_DIR = os.environ.get("BASE_DIR", ".")
OUTPUTS_ROOT = os.path.join(BASE_DIR, "outputs")

def find_error_samples_for_model(predictions_path):
    """
    Analyze one model prediction file and save false-positive (FP) and false-negative (FN) samples.

    Args:
        predictions_path (str): Path to the test-set prediction CSV file.
    """
    if not os.path.exists(predictions_path):
        print(f"  Warning: prediction file not found, skipping: {predictions_path}")
        return

    print(f"  Analyzing: {os.path.basename(predictions_path)}")

    # 1. Load prediction results.
    try:
        df_pred = pd.read_csv(predictions_path)
    except Exception as e:
        print(f"    Error loading file: {e}")
        return

    # 2. Select error samples.
    #    False positives (FP): predicted positive but actually negative.
    false_positives = df_pred[(df_pred['y_true'] == 0) & (df_pred['pred_label'] == 1)]

    #    False negatives (FN): predicted negative but actually positive.
    false_negatives = df_pred[(df_pred['y_true'] == 1) & (df_pred['pred_label'] == 0)]

    # 3. Save results.
    output_dir = os.path.dirname(predictions_path)
    fp_path = os.path.join(output_dir, "false_positive_samples.csv")
    fn_path = os.path.join(output_dir, "false_negative_samples.csv")

    false_positives.to_csv(fp_path, index=False, encoding='utf-8-sig')
    false_negatives.to_csv(fn_path, index=False, encoding='utf-8-sig')

    print(f"    False-positive (FP) samples: {len(false_positives)} rows -> saved to {fp_path}")
    print(f"    False-negative (FN) samples: {len(false_negatives)} rows -> saved to {fn_path}")


def main():
    """Run error analysis over all model test prediction files."""
    print(f"Searching for test prediction files under: {OUTPUTS_ROOT}")

    # Find all files named 'test_predictions_*.csv'.
    # The wildcard '*' matches different model identifiers.
    search_pattern = os.path.join(OUTPUTS_ROOT, "*", "test_predictions_*.csv")
    all_pred_files = glob.glob(search_pattern)

    if not all_pred_files:
        print("Error: no test prediction files were found (test_predictions_*.csv).")
        print("Please check whether BASE_DIR and OUTPUTS_ROOT are correct.")
        return

    print(f"Found {len(all_pred_files)} prediction files. Starting analysis...\n")
    for pred_file in all_pred_files:
        find_error_samples_for_model(pred_file)

    print("\nError analysis completed for all models.")

if __name__ == "__main__":
    main()


## 7. Exploratory Keyword Discovery with TF-IDF: Article Titles

This section uses TF-IDF and distinction scores to identify title keywords that are more associated with Page 1–3 priority articles. These results were used as an exploratory step before manual keyword curation.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import jieba
import pandas as pd
import os

# Configure paths.
# Point directly to the existing training split file.
TRAIN_DATA_PATH = os.path.join(os.environ.get("BASE_DIR", "."), "train_df.parquet")
TEXT_COL = "title"
LABEL_COL = "labels"

# 1. Load data.
print(f"Loading data from {TRAIN_DATA_PATH}...")
train_df = pd.read_parquet(TRAIN_DATA_PATH)

# 2. Run TF-IDF analysis.
print(f"Analyzing training-set title keywords. Number of samples: {len(train_df)}")

# Tokenize Chinese text with jieba.
pos_texts = train_df[train_df[LABEL_COL] == 1][TEXT_COL].apply(lambda x: " ".join(jieba.cut(str(x))))
neg_texts = train_df[train_df[LABEL_COL] == 0][TEXT_COL].apply(lambda x: " ".join(jieba.cut(str(x))))

# 3. Compute TF-IDF.
vectorizer = TfidfVectorizer(max_features=1000)
tfidf_pos = vectorizer.fit_transform(pos_texts)
words = vectorizer.get_feature_names_out()
scores_pos = tfidf_pos.mean(axis=0).A1

# Apply the same vectorizer to non-priority articles for comparison.
tfidf_neg = vectorizer.transform(neg_texts)
scores_neg = tfidf_neg.mean(axis=0).A1

# 4. Build the comparison table.
analysis_df = pd.DataFrame({
    'keyword': words,
    'pos_tfidf_priority': scores_pos,
    'neg_tfidf_non_priority': scores_neg
})

# Compute distinction score: high priority TF-IDF and low non-priority TF-IDF indicate stronger discrimination.
analysis_df['diff_score'] = analysis_df['pos_tfidf_priority'] / (analysis_df['neg_tfidf_non_priority'] + 1e-9)

print("Analysis completed. Run the next cell to inspect the results.")


## 8. Inspect High-Distinction Title Keywords

This cell displays the strongest title keywords by distinction score and applies simple filtering rules to identify candidate feature keywords.


In [ ]:
# Select the keywords with the highest distinction scores.
top_20_diff = analysis_df.sort_values(by='diff_score', ascending=False).head(30)

print("--- Top TF-IDF keywords by distinction score ---")
print("Note: diff_score = priority TF-IDF / non-priority TF-IDF")
display(top_20_diff)
# Filtering rule: pos_tfidf > 0.005 to remove very rare words; diff_score > 5 to ensure discrimination.
filtered_keywords = analysis_df[
    (analysis_df['pos_tfidf_priority'] > 0.005) &
    (analysis_df['diff_score'] > 5)
].sort_values(by='diff_score', ascending=False)

print(f"--- Filtered high-quality feature keywords: {len(filtered_keywords)} total ---")
display(filtered_keywords.head(30))


## 9. Manual Keyword Check: Title Distribution

Use this cell to manually inspect how a selected keyword is distributed between priority and non-priority articles.


In [ ]:
target_word = "硬仗"
pos_samples = train_df[(train_df[LABEL_COL] == 1) & (train_df[TEXT_COL].str.contains(target_word))]
neg_samples = train_df[(train_df[LABEL_COL] == 0) & (train_df[TEXT_COL].str.contains(target_word))]

print(f"Distribution for keyword '{target_word}':")
print(f"Priority-class occurrences: {len(pos_samples)}")
print(f"Non-priority occurrences: {len(neg_samples)}")

if len(pos_samples) > 0:
    print("\nPriority-class sample titles:")
    display(pos_samples[TEXT_COL].head(10))


## 10. Exploratory Keyword Discovery with TF-IDF: Article Bodies

This section repeats the TF-IDF distinction analysis on article body text.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import jieba
import pandas as pd
import os

# Configure paths.
# Point directly to the existing training split file.
TRAIN_DATA_PATH = os.path.join(os.environ.get("BASE_DIR", "."), "train_df.parquet")
CONTENT_COL = "body"
LABEL_COL = "labels"

# 1. Load data.
print(f"Loading data from {TRAIN_DATA_PATH}...")
train_df = pd.read_parquet(TRAIN_DATA_PATH)

# Run TF-IDF analysis.
print(f"Analyzing training-set body text. Number of samples: {len(train_df)}")

# 2. Tokenize Chinese text.
pos_texts = train_df[train_df[LABEL_COL] == 1][CONTENT_COL].apply(lambda x: " ".join(jieba.cut(str(x))))
neg_texts = train_df[train_df[LABEL_COL] == 0][CONTENT_COL].apply(lambda x: " ".join(jieba.cut(str(x))))

# 3. Compute TF-IDF for priority-class body text.
# Body text has a larger vocabulary, so max_features is increased to 2000.
vectorizer = TfidfVectorizer(max_features=2000)
tfidf_pos = vectorizer.fit_transform(pos_texts)
words = vectorizer.get_feature_names_out()
scores_pos = tfidf_pos.mean(axis=0).A1

# 4. Evaluate the same terms in non-priority body text.
tfidf_neg = vectorizer.transform(neg_texts)
scores_neg = tfidf_neg.mean(axis=0).A1

# 5. Build the comparison table.
analysis_df = pd.DataFrame({
    'keyword': words,
    'pos_tfidf_priority_body': scores_pos,
    'neg_tfidf_non_priority_body': scores_neg
})

analysis_df['diff_score'] = analysis_df['pos_tfidf_priority_body'] / (analysis_df['neg_tfidf_non_priority_body'] + 1e-9)

print("\n--- Priority-class body keyword distinction analysis ---")
result = analysis_df.sort_values(by='pos_tfidf_priority_body', ascending=False).head(50)
display(result)

print("\nNote: higher diff_score means the term is more specific to priority-class body text.")


## 11. Inspect High-Distinction Body Keywords

This cell displays the top body keywords by distinction score.


In [ ]:
# Select the top 30 keywords by distinction score.
top_30_diff = analysis_df.sort_values(by='diff_score', ascending=False).head(30)

print("--- Top body TF-IDF keywords by distinction score ---")
print("Note: diff_score = priority body TF-IDF / non-priority body TF-IDF")
display(top_30_diff)


## 12. Filter Candidate Body Keywords

This cell applies frequency and distinction-score thresholds to select candidate body keywords for manual review.


In [ ]:
# Identify the correct column names from analysis_df
pos_col = 'pos_tfidf_priority_body'
diff_col = 'diff_score'

# Quick safety check: if the column name is slightly different, find the one starting with 'pos_tfidf'
if pos_col not in analysis_df.columns:
    matched_cols = [c for c in analysis_df.columns if c.startswith('pos_tfidf')]
    if matched_cols:
        pos_col = matched_cols[0]

# Filtering rule: pos_tfidf > 0.005 to remove rare body terms; diff_score > 3 to ensure discrimination.
filtered_keywords_body = analysis_df[
    (analysis_df[pos_col] > 0.005) &
    (analysis_df[diff_col] > 3)
].sort_values(by=diff_col, ascending=False)

print(f"--- Filtered high-quality body feature keywords: {len(filtered_keywords_body)} total ---")
display(filtered_keywords_body.head(30))


## 13. Manual Keyword Check: Body Distribution

Use this cell to manually inspect how a selected body keyword is distributed between priority and non-priority articles.


In [ ]:
target_word = "优秀党员"
# Ensure columns are defined
TEXT_COL = "title"
CONTENT_COL = "body"

# Use CONTENT_COL to inspect this keyword distribution in body text.
pos_samples = train_df[(train_df[LABEL_COL] == 1) & (train_df[CONTENT_COL].str.contains(target_word, na=False))]
neg_samples = train_df[(train_df[LABEL_COL] == 0) & (train_df[CONTENT_COL].str.contains(target_word, na=False))]

print(f"Body keyword distribution for '{target_word}':")
print(f"Priority-class body occurrences: {len(pos_samples)}")
print(f"Non-priority body occurrences: {len(neg_samples)}")

if len(pos_samples) > 0:
    print(f"\nPriority-class article titles containing '{target_word}':")
    display(pos_samples[TEXT_COL].head(10))


## 14. Keyword Distinction Weighting

This section converts distinction ratios into keyword weights using `weight = log2(ratio)`. This can help rank manually reviewed candidate keywords by how strongly they separate priority and non-priority articles.


In [ ]:
import pandas as pd
import jieba
from collections import Counter
import os

TRAIN_DATA_PATH = os.path.join(os.environ.get("BASE_DIR", "."), "train_df.parquet")
if os.path.exists(TRAIN_DATA_PATH):
    train_df = pd.read_parquet(TRAIN_DATA_PATH)
else:
    print(f"Error: File not found at {TRAIN_DATA_PATH}")

def extract_top_distinction_keywords(df, text_col, label_col='labels', top_n=50, min_ratio=5.0, min_pos_count=5):
    # 1. Separate priority and non-priority texts.
    pos_texts = df[df[label_col] == 1][text_col].astype(str)
    neg_texts = df[df[label_col] == 0][text_col].astype(str)

    pos_total = len(pos_texts)
    neg_total = len(neg_texts)

    # 2. Tokenize and count document-level word occurrences.
    def get_word_counts(texts):
        counts = Counter()
        for t in texts:
            words = set(jieba.cut(t))  # Use set() to count whether each article contains the word.
            counts.update(words)
        return counts

    pos_counts = get_word_counts(pos_texts)
    neg_counts = get_word_counts(neg_texts)

    # 3. Compute distinction ratio.
    stats = []
    for word, p_count in pos_counts.items():
        if p_count < min_pos_count: continue  # Filter out words with too few priority-class occurrences.

        p_rate = p_count / pos_total
        n_count = neg_counts.get(word, 0)
        n_rate = n_count / neg_total

        ratio = p_rate / (n_rate + 1e-9)

        stats.append({
            'keyword': word,
            'pos_count': p_count,
            'neg_count': n_count,
            'pos_rate': p_rate,
            'distinction_ratio': ratio
        })

    stats_df = pd.DataFrame(stats)
    # Keep words that are both distinctive and sufficiently frequent in the priority class.
    gold_features = stats_df[stats_df['distinction_ratio'] >= min_ratio].sort_values('distinction_ratio', ascending=False)

    return gold_features

# Run analysis.
if 'train_df' in locals():
    print("--- Analyzing distinctive title keywords ---")
    title_gold = extract_top_distinction_keywords(train_df, 'title')
    display(title_gold.head(20))

    print("\n--- Analyzing distinctive body keywords ---")
    # Body analysis can be slower; this scans the available samples directly.
    body_gold = extract_top_distinction_keywords(train_df, 'body', min_pos_count=10)
    display(body_gold.head(20))
    output_path = '/content/drive/MyDrive/Colab Notebooks/title_gold_keywords.xlsx'
    title_gold.to_excel(output_path, index=False)
    output_path = '/content/drive/MyDrive/Colab Notebooks/body_gold_keywords.xlsx'
    body_gold.to_excel(output_path, index=False)
